In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys
import logging
logging.basicConfig(level=logging.INFO)
print(sys.executable)

/home/jop9552/miniconda3/envs/mmdeploy_JP_v2/bin/python


In [3]:
from multicamera_airflow_pipeline.jonah_241112.keypoints.tensorrt import RTMModelConverter

INFO:multicamera_airflow_pipeline.jonah_241112.keypoints.tensorrt:Python interpreter binary location: /home/jop9552/miniconda3/envs/mmdeploy_JP_v2/bin/python


In [4]:
# https://o2portal.rc.hms.harvard.edu/node/compute-g-17-148.o2.rc.hms.harvard.edu/59141/notebooks/tim_sainburg/projects/23-09-29-peromoseq/notebooks/keypoints/mmpose/24-01-11-deploy-2d-predictions-on-local/24-05-08-convert-pose-model.ipynb
# rtmpose_model_name = 'rtmpose-m_8xb64-210e_ap10k-256x256_24-05-04-21-35-13_305524'
# path_to_rmpose_config = Path('/n/groups/datta/tim_sainburg/projects/24-01-05-multicamera_keypoints_mm2d/models/rtmpose/rtmpose-m_8xb64-210e_ap10k-256x256_24-05-04-21-35-13_305524/config.py')
# path_to_rmpose_checkpoint = Path('/n/groups/datta/tim_sainburg/projects/24-01-05-multicamera_keypoints_mm2d/models/rtmpose/rtmpose-m_8xb64-210e_ap10k-256x256_24-05-04-21-35-13_305524/best_PCK_epoch_200.pth')

rtmpose_model_name = "rtmpose-m_8xb64-210e_ap10k-256x256_24-11-01-20-36-55"
path_to_rmpose_config = Path("/n/groups/datta/6cam_keypoint_networks/mm_pose/Jonah/20241030_v1/rtmpose/rtmpose-m_8xb64-210e_ap10k-256x256_24-11-01-20-36-55/config.py")
path_to_rmpose_checkpoint = path_to_rmpose_config.parent / "epoch_90.pth"
skeleton_py_file='/n/groups/datta/Jonah/Local_code_groups/6cam_repos/multicam_airflow_pipeline/multicamera_airflow_pipeline/jonah_241112/skeletons/weinreb15.py'

In [5]:
# https://o2portal.rc.hms.harvard.edu/node/compute-g-17-148.o2.rc.hms.harvard.edu/59141/notebooks/tim_sainburg/projects/23-09-29-peromoseq/notebooks/keypoints/mmpose/24-01-11-deploy-2d-predictions-on-local/240508-convert-detection-model-fp32.ipynb
# rtmdetection_model_name = 'rtmdet_tiny_8xb32-300e_coco_chronic_24-05-04-17-51-58_216661'
# path_to_rtmdetection_config = Path('/n/groups/datta/tim_sainburg/projects/24-01-05-multicamera_keypoints_mm2d/models/rtmdet/rtmdet_tiny_8xb32-300e_coco_chronic_24-05-04-17-51-58_216661/config.py')
# path_to_rtmdetection_checkpoint = Path('/n/groups/datta/tim_sainburg/projects/24-01-05-multicamera_keypoints_mm2d/models/rtmdet/rtmdet_tiny_8xb32-300e_coco_chronic_24-05-04-17-51-58_216661/epoch_400.pth')

rtmdetection_model_name = "rtmdet_small_8xb32-300e_coco_chronic_24-10-31-12-48-26"
path_to_rtmdetection_config = Path("/n/groups/datta/6cam_keypoint_networks/mm_pose/Jonah/20241030_v1/rtmdet/rtmdet_small_8xb32-300e_coco_chronic_24-10-31-12-48-26/config.py")
path_to_rtmdetection_checkpoint = path_to_rtmdetection_config.parent / "epoch_63.pth"


In [13]:
output_directory_tensorrt= "/n/groups/datta/kpts_pipeline/jonah_241112/results/tensorrt"
conda_env='/home/jop9552/miniconda3/envs/mmdeploy_JP_v2'
path_to_mmdeploy='/n/groups/datta/Jonah/Local_code_groups/tensorrt_install/mmdeploy'


In [7]:
model_converter = RTMModelConverter(
    path_to_rmpose_config = path_to_rmpose_config,
    path_to_rmpose_checkpoint = path_to_rmpose_checkpoint,
    path_to_rtmdetection_config = path_to_rtmdetection_config,
    path_to_rtmdetection_checkpoint = path_to_rtmdetection_checkpoint,
    tensorrt_output_directory = output_directory_tensorrt,
    rtmdetection_model_name = rtmdetection_model_name,
    rtmpose_model_name = rtmpose_model_name,
    # TODO: change these defaults from tim's to mine
    conda_env = conda_env,
    path_to_mmdeploy = path_to_mmdeploy,
    skeleton_py_file = skeleton_py_file,
)

INFO:multicamera_airflow_pipeline.jonah_241112.keypoints.tensorrt:Using CUDA device: NVIDIA_A40


In [8]:
model_converter.run()

INFO:multicamera_airflow_pipeline.jonah_241112.keypoints.tensorrt:Converting detection model to tensorrt. input: /n/groups/datta/kpts_pipeline/jonah_241112/results/tensorrt_testing/rtmdet_small_8xb32-300e_coco_chronic_24-10-31-12-48-26/NVIDIA_A40
INFO:multicamera_airflow_pipeline.jonah_241112.keypoints.tensorrt:Checking if tensorrt model exists at: /n/groups/datta/kpts_pipeline/jonah_241112/results/tensorrt_testing/rtmdet_small_8xb32-300e_coco_chronic_24-10-31-12-48-26/NVIDIA_A40


Running model conversion script:
module load gcc/9.2.0
module load cuda/11.7
TENSORRT_DIR=/n/groups/datta/Jonah/Local_code_groups/tensorrt_install/TensorRT-8.6.1.6
export LD_LIBRARY_PATH=${TENSORRT_DIR}/lib:$LD_LIBRARY_PATH
eval "$(conda shell.bash hook)";
conda activate /home/jop9552/miniconda3/envs/mmdeploy_JP_v2;
export PYTHONPATH=/tmp/tmpnld47jlu:$PYTHONPATH;
python /n/groups/datta/Jonah/Local_code_groups/tensorrt_install/mmdeploy/tools/deploy.py /n/groups/datta/tim_sainburg/projects/mmdeploy/configs/mmdet/detection/detection_tensorrt_static-320x320.py /n/groups/datta/6cam_keypoint_networks/mm_pose/Jonah/20241030_v1/rtmdet/rtmdet_small_8xb32-300e_coco_chronic_24-10-31-12-48-26/config.py /n/groups/datta/6cam_keypoint_networks/mm_pose/Jonah/20241030_v1/rtmdet/rtmdet_small_8xb32-300e_coco_chronic_24-10-31-12-48-26/epoch_63.pth /n/groups/datta/tim_sainburg/projects/24-01-05-multicamera_keypoints_mm2d/example_data/test_mouse.png --work-dir /n/groups/datta/kpts_pipeline/jonah_241112/resu

INFO:multicamera_airflow_pipeline.jonah_241112.keypoints.tensorrt:Finished converting detection model to tensorrt.
INFO:multicamera_airflow_pipeline.jonah_241112.keypoints.tensorrt:model output at: /n/groups/datta/kpts_pipeline/jonah_241112/results/tensorrt_testing/rtmdet_small_8xb32-300e_coco_chronic_24-10-31-12-48-26/NVIDIA_A40
INFO:multicamera_airflow_pipeline.jonah_241112.keypoints.tensorrt:Converting pose model to tensorrt. input: /n/groups/datta/kpts_pipeline/jonah_241112/results/tensorrt_testing/rtmpose-m_8xb64-210e_ap10k-256x256_24-11-01-20-36-55/NVIDIA_A40
INFO:multicamera_airflow_pipeline.jonah_241112.keypoints.tensorrt:Checking if tensorrt model exists at: /n/groups/datta/kpts_pipeline/jonah_241112/results/tensorrt_testing/rtmpose-m_8xb64-210e_ap10k-256x256_24-11-01-20-36-55/NVIDIA_A40


module load gcc/9.2.0
module load cuda/11.7
TENSORRT_DIR=/n/groups/datta/Jonah/Local_code_groups/tensorrt_install/TensorRT-8.6.1.6
export LD_LIBRARY_PATH=${TENSORRT_DIR}/lib:$LD_LIBRARY_PATH
eval "$(conda shell.bash hook)";
conda activate /home/jop9552/miniconda3/envs/mmdeploy_JP_v2;
export PYTHONPATH=/tmp/tmpa_uuoti_:$PYTHONPATH;
python /n/groups/datta/Jonah/Local_code_groups/tensorrt_install/mmdeploy/tools/deploy.py /n/groups/datta/tim_sainburg/projects/mmdeploy/configs/mmpose/pose-detection_simcc_tensorrt_dynamic-256x256.py /n/groups/datta/6cam_keypoint_networks/mm_pose/Jonah/20241030_v1/rtmpose/rtmpose-m_8xb64-210e_ap10k-256x256_24-11-01-20-36-55/config.py /n/groups/datta/6cam_keypoint_networks/mm_pose/Jonah/20241030_v1/rtmpose/rtmpose-m_8xb64-210e_ap10k-256x256_24-11-01-20-36-55/epoch_90.pth /n/groups/datta/tim_sainburg/projects/24-01-05-multicamera_keypoints_mm2d/example_data/test_mouse_cropped.png --work-dir /n/groups/datta/kpts_pipeline/jonah_241112/results/tensorrt_testing/rtm

INFO:multicamera_airflow_pipeline.jonah_241112.keypoints.tensorrt:Finished converting pose model to tensorrt.
INFO:multicamera_airflow_pipeline.jonah_241112.keypoints.tensorrt:model output at: /n/groups/datta/kpts_pipeline/jonah_241112/results/tensorrt_testing/rtmpose-m_8xb64-210e_ap10k-256x256_24-11-01-20-36-55/NVIDIA_A40


In [9]:
!ls {output_directory_tensorrt.as_posix()}

rtmdet_small_8xb32-300e_coco_chronic_24-10-31-12-48-26
rtmpose-m_8xb64-210e_ap10k-256x256_24-11-01-20-36-55


In [10]:
output_directory_tensorrt

PosixPath('/n/groups/datta/kpts_pipeline/jonah_241112/results/tensorrt_testing')

In [11]:
!ls {output_directory_tensorrt / "rtmdet_tiny_8xb32-300e_coco_chronic_24-05-04-17-51-58_216661/NVIDIA_L40S"}

ls: cannot access /n/groups/datta/kpts_pipeline/jonah_241112/results/tensorrt_testing/rtmdet_tiny_8xb32-300e_coco_chronic_24-05-04-17-51-58_216661/NVIDIA_L40S: No such file or directory


In [12]:
!ls {output_directory_tensorrt / "rtmpose-m_8xb64-210e_ap10k-256x256_24-05-04-21-35-13_305524/NVIDIA_L40S"}

ls: cannot access /n/groups/datta/kpts_pipeline/jonah_241112/results/tensorrt_testing/rtmpose-m_8xb64-210e_ap10k-256x256_24-05-04-21-35-13_305524/NVIDIA_L40S: No such file or directory
